# 08 — Artifact 5: Creatinine Cross-Format Merge

🔧 Script · 📚 Reference

**Artifact 5:** A creatinine result (1.4 mg/dL on 2025-09-12) appears in two sources — the Epic EHI SQLite dump and the synthesized lab PDF. Both express LOINC 2160-0. Layer 3 deduplicates them into one merged Observation with both source-tags, both identifiers, and the max quality score.


## 1. Build the two silver-tier Observations

In [ ]:
from ehi_atlas.harmonize.provenance import SYS_SOURCE_TAG, SYS_LIFECYCLE

obs_epic = {
    "resourceType": "Observation",
    "id": "epic-obs-creatinine",
    "meta": {
        "tag": [
            {"system": SYS_SOURCE_TAG, "code": "epic-ehi"},
            {"system": SYS_LIFECYCLE,  "code": "stub-silver"},
        ],
        "extension": [{"url": "https://ehi-atlas.example/fhir/StructureDefinition/quality-score",
                        "valueDecimal": 0.78}],
    },
    "status": "final",
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category",
                               "code": "laboratory"}]}],
    "code": {
        "coding": [{"system": "http://loinc.org", "code": "2160-0",
                    "display": "Creatinine [Mass/volume] in Serum or Plasma"}],
        "text": "Creatinine",
    },
    "subject": {"reference": "Patient/rhett759"},
    "effectiveDateTime": "2025-09-12",
    "valueQuantity": {"value": 1.4, "unit": "mg/dL", "system": "http://unitsofmeasure.org", "code": "mg/dL"},
}

obs_lab_pdf = {
    "resourceType": "Observation",
    "id": "lab-pdf-obs-creatinine-rhett759",
    "meta": {
        "tag": [
            {"system": SYS_SOURCE_TAG, "code": "lab-pdf"},
            {"system": SYS_LIFECYCLE,  "code": "stub-silver"},
        ],
        "extension": [{"url": "https://ehi-atlas.example/fhir/StructureDefinition/quality-score",
                        "valueDecimal": 0.48}],
    },
    "status": "final",
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category",
                               "code": "laboratory"}]}],
    "code": {
        "coding": [{"system": "http://loinc.org", "code": "2160-0",
                    "display": "Creatinine [Mass/volume] in Serum or Plasma"}],
        "text": "Creatinine",
    },
    "subject": {"reference": "Patient/rhett759"},
    "effectiveDateTime": "2025-09-12",
    "valueQuantity": {"value": 1.4, "unit": "mg/dL", "system": "http://unitsofmeasure.org", "code": "mg/dL"},
}

print("Built obs_epic (quality 0.78) and obs_lab_pdf (quality 0.48)")


## 2. extract_observation_key

In [ ]:
# 🔧 Script
from ehi_atlas.harmonize.observation import extract_observation_key

key_epic    = extract_observation_key(obs_epic)
key_lab_pdf = extract_observation_key(obs_lab_pdf)

print("Epic key:   ", key_epic)
print("Lab-PDF key:", key_lab_pdf)


## 3. observations_equivalent → True

In [ ]:
from ehi_atlas.harmonize.observation import observations_equivalent

equivalent = observations_equivalent(obs_epic, obs_lab_pdf)
print("observations_equivalent(obs_epic, obs_lab_pdf):", equivalent)
print("→ Same LOINC + same date + same value (±0.1%) + same unit → exact dedup")


## 4. merge_observations

In [ ]:
from ehi_atlas.harmonize.observation import merge_observations

result = merge_observations([obs_epic, obs_lab_pdf], "merged-obs-rhett-creatinine")
merged = result.merged

print("Merged Observation ID:", merged["id"])
print("Sources:", result.sources)
print("Rationale:", result.rationale)


## 5. Inspect the merged Observation

In [ ]:
import json

print("status:", merged["status"])
print("effectiveDateTime:", merged.get("effectiveDateTime"))
print("valueQuantity:", merged.get("valueQuantity"))

print()
print("meta.tag (both source-tags):")
for tag in merged.get("meta", {}).get("tag", []):
    print(f"  {tag.get('system','').split('/')[-1]}: {tag.get('code','')}")

print()
quality_ext = next(
    (e for e in merged.get("meta", {}).get("extension", [])
     if "quality-score" in e.get("url", "")),
    None
)
print("quality score (max of 0.78, 0.48):", quality_ext.get("valueDecimal") if quality_ext else "—")

print()
rationale_ext = next(
    (e for e in merged.get("meta", {}).get("extension", [])
     if "merge-rationale" in e.get("url", "")),
    None
)
print("merge-rationale:", rationale_ext.get("valueString") if rationale_ext else "—")


## 6. Confirm in the actual gold bundle

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
gold_bundle = json.loads(
    (ATLAS_ROOT / "corpus" / "gold" / "patients" / "rhett759" / "bundle.json").read_text()
)

creatinine_obs = None
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "Observation":
        continue
    codings = res.get("code", {}).get("coding", [])
    if any(c.get("code") == "2160-0" for c in codings):
        creatinine_obs = res
        break

if creatinine_obs:
    print("Gold creatinine Observation:", creatinine_obs["id"])
    print("effectiveDateTime:", creatinine_obs.get("effectiveDateTime"))
    print("valueQuantity:", creatinine_obs.get("valueQuantity"))
    src_tags = [t["code"] for t in creatinine_obs.get("meta", {}).get("tag", [])
                if "source-tag" in t.get("system", "")]
    print("source-tags:", src_tags)
else:
    print("Not found — re-run 'make pipeline' to regenerate gold tier")


**Next:** [09_orchestrator_end_to_end.ipynb](./09_orchestrator_end_to_end.ipynb)